In [0]:
# dbutils.fs.rm(
#     "abfss://silver@emedallionstore01.dfs.core.windows.net/delta/carts",
#     True
# )
# spark.sql(f"""DROP TABLE IF EXISTS emain_databricks.silver.carts""")
# dbutils.fs.rm(
#     "abfss://silver@emedallionstore01.dfs.core.windows.net/delta/products",
#     True
# )
# spark.sql(f"""DROP TABLE IF EXISTS emain_databricks.silver.products""")
# dbutils.fs.rm(
#     "abfss://silver@emedallionstore01.dfs.core.windows.net/delta/users",
#     True
# )
# spark.sql(f"""DROP TABLE IF EXISTS emain_databricks.silver.users""")

In [0]:
TABLE_CONFIG={
    "products":{
        "source":"emain_databricks.bronze.products",
        "target":"emain_databricks.silver.products",
        "primary_key":"id",
        "delta_path":"abfss://silver@emedallionstore01.dfs.core.windows.net/delta/products"
    },

    "users":{
        "source":"emain_databricks.bronze.users",
        "target":"emain_databricks.silver.users",
        "primary_key":"id",
        "delta_path":"abfss://silver@emedallionstore01.dfs.core.windows.net/delta/users"
    },

    "carts":{
        "source":"emain_databricks.bronze.carts",
        "target":"emain_databricks.silver.carts",
        "primary_key":"id",
        "delta_path":"abfss://silver@emedallionstore01.dfs.core.windows.net/delta/carts"
    }
}

In [0]:
from pyspark.sql.functions import col

SCHEMA_MAPPING = {

    "products": {
        "id": "int",
        "price": "double",
        "stock": "int",
        "rating": "double",
        "discountPercentage": "double",
        "weight": "double",
        "minimumOrderQuantity": "int"
    },

    "users": {
        "id": "int",
        "age": "int",
        "height": "double",
        "weight": "double"
    },

    "carts": {
        "id": "int",
        "userId": "int",
        "total": "double",
        "discountedTotal": "double",
        "totalProducts": "int",
        "totalQuantity": "int"
    }

}

In [0]:
from pyspark.sql.functions import col

def apply_schema(df, dataset):

    schema = SCHEMA_MAPPING[dataset]

    for column_name, datatype in schema.items():

        df = df.withColumn(
            column_name,
            col(column_name).cast(datatype)
        )

    return df

In [0]:
from pyspark.sql.functions import col

VALIDATION_RULES={

    "products":(
        (col("price")>=0)&
        (col("stock")>=0)&
        (col("rating").between(0,5))&
        (col("title").isNotNull())&
        (col("category").isNotNull())
    ),

    "users":(
        (col("age")>0)&
        (col("email").isNotNull())&
        (col("firstName").isNotNull())&
        (col("lastName").isNotNull())
    ),

    "carts":(
        (col("total")>=0)&
        (col("totalQuantity")>0)&
        (col("discountedTotal")<=col("total"))
    )

}





In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number,current_timestamp,lit
from delta.tables import DeltaTable

def process_table(dataset): 
    config=TABLE_CONFIG[dataset]
    print("=" * 60)
    print(f"Processing {dataset.upper()}")
    print("=" * 60)
    bronze_df=spark.table(config["source"])
    
    windows_spec=(Window.partitionBy(config["primary_key"]).orderBy(col("ingestion_timestamp").desc()))
    silver_df=(
        bronze_df.withColumn("row_num",row_number().over(windows_spec)).filter(col("row_num")==1).drop("row_num")
    )
    rows_read = bronze_df.count()
    
    
    
    rule=VALIDATION_RULES[dataset]
    valid_df=silver_df.filter(rule)
    invalid_df=silver_df.filter(~rule)
    rows_valid = valid_df.count()
    rows_invalid = invalid_df.count()

    valid_df= (
        valid_df.withColumn("processing_timestamp",current_timestamp()).withColumn("pipeline_name",lit("silver_pipeline"))
    )


    
    if not DeltaTable.isDeltaTable(spark, config["delta_path"]):

        (
            valid_df.write
            .format("delta")
            .mode("overwrite")
            .save(config["delta_path"])
        )
        print(f"{dataset.upper()} Initial Delta Table Created")

    else:

        print(f"{dataset.upper()} Delta Table Already Exists")

    spark.sql(f"""
              create table if not exists {config["target"]} using delta location '{config["delta_path"]}' """)
    
    delta_table=DeltaTable.forPath(spark,config["delta_path"])
    target=delta_table.alias("target")
    source=valid_df.alias("source")
    merge_condition = (
    f"target.{config['primary_key']} = source.{config['primary_key']}"
    )

    (target.merge(source=source,condition=merge_condition).whenMatchedUpdateAll()
     .whenNotMatchedInsertAll().execute()
    )
    print(f"Rows Read      : {rows_read}")
    print(f"Rows Valid     : {rows_valid}")
    print(f"Rows Invalid   : {rows_invalid}")
    print(f"Rows Written   : {rows_valid}")
    print("=" * 60)
    print(f"{dataset.upper()} completed successfully\n")
    print(f"{dataset.upper()} completed successfully\n")
    invalid_df.display()
    
    




In [0]:
def main():
    for dataset in TABLE_CONFIG:
        process_table(dataset)

    print("=" * 60)
    print("SILVER PIPELINE COMPLETED")
    print("=" * 60)

main()

Processing PRODUCTS
PRODUCTS Initial Delta Table Created
Rows Read      : 4074
Rows Valid     : 194
Rows Invalid   : 0
Rows Written   : 194
PRODUCTS completed successfully

PRODUCTS completed successfully



availabilityStatus,brand,category,description,dimensions,discountPercentage,id,images,ingestion_timestamp,meta,minimumOrderQuantity,price,rating,returnPolicy,reviews,shippingInformation,sku,stock,tags,thumbnail,title,warrantyInformation,weight,_rescued_data


Processing USERS
USERS Initial Delta Table Created
Rows Read      : 4368
Rows Valid     : 208
Rows Invalid   : 0
Rows Written   : 208
USERS completed successfully

USERS completed successfully



address,age,bank,birthDate,bloodGroup,company,crypto,ein,email,eyeColor,firstName,gender,hair,height,id,image,ingestion_timestamp,ip,lastName,macAddress,maidenName,password,phone,role,ssn,university,userAgent,username,weight,_rescued_data


Processing CARTS
CARTS Initial Delta Table Created
Rows Read      : 4368
Rows Valid     : 195
Rows Invalid   : 13
Rows Written   : 195
CARTS completed successfully

CARTS completed successfully



discountedTotal,id,ingestion_timestamp,products,total,totalProducts,totalQuantity,userId,_rescued_data
381.68,15,2026-07-02T08:27:27.582966,"List(List(17.02, 58.07, 61, 34.99, 2, https://cdn.dummyjson.com/product-images/kitchen-accessories/hand-blender/thumbnail.webp, Hand Blender, 69.98), List(17.17, 82.82, 105, 99.99, 1, https://cdn.dummyjson.com/product-images/mobile-accessories/apple-magsafe-battery-pack/thumbnail.webp, Apple MagSafe Battery Pack, 99.99), List(7.74, 73.79, 141, 39.99, 2, https://cdn.dummyjson.com/product-images/sports-accessories/basketball-rim/thumbnail.webp, Basketball Rim, 79.98))",249.95,3,5,15,null
281.89,48,2026-07-02T08:27:27.582966,"List(List(7.44, 69.37, 140, 14.99, 5, https://cdn.dummyjson.com/product-images/sports-accessories/basketball/thumbnail.webp, Basketball, 74.95), List(9.58, 32.52, 18, 8.99, 4, https://cdn.dummyjson.com/product-images/groceries/cat-food/thumbnail.webp, Cat Food, 35.96))",110.91,2,11,48,null
330.77,72,2026-07-02T08:27:27.582966,"List(List(7.74, 147.58, 141, 39.99, 4, https://cdn.dummyjson.com/product-images/sports-accessories/basketball-rim/thumbnail.webp, Basketball Rim, 159.96), List(6.84, 69.82, 46, 14.99, 5, https://cdn.dummyjson.com/product-images/home-decoration/plant-pot/thumbnail.webp, Plant Pot, 74.95), List(17.02, 58.07, 61, 34.99, 2, https://cdn.dummyjson.com/product-images/kitchen-accessories/hand-blender/thumbnail.webp, Hand Blender, 69.98), List(9.29, 16.3, 38, 5.99, 3, https://cdn.dummyjson.com/product-images/groceries/rice/thumbnail.webp, Rice, 17.97))",322.86,4,14,72,null
182.13,102,2026-07-02T08:27:27.582966,"List(List(10.27, 9.86, 22, 10.99, 1, https://cdn.dummyjson.com/product-images/groceries/dog-food/thumbnail.webp, Dog Food, 10.99), List(19.12, 10.51, 111, 12.99, 1, https://cdn.dummyjson.com/product-images/mobile-accessories/selfie-stick-monopod/thumbnail.webp, Selfie Stick Monopod, 12.99), List(8.03, 47.79, 53, 12.99, 4, https://cdn.dummyjson.com/product-images/kitchen-accessories/chopping-board/thumbnail.webp, Chopping Board, 51.96), List(7.46, 74.01, 45, 39.99, 2, https://cdn.dummyjson.com/product-images/home-decoration/house-showpiece-plant/thumbnail.webp, House Showpiece Plant, 79.98), List(18.91, 12.96, 63, 7.99, 2, https://cdn.dummyjson.com/product-images/kitchen-accessories/kitchen-sieve/thumbnail.webp, Kitchen Sieve, 15.98))",171.9,5,10,102,null
340.15999999999997,119,2026-07-02T08:27:27.582966,"List(List(18.52, 32.58, 104, 19.99, 2, https://cdn.dummyjson.com/product-images/mobile-accessories/apple-iphone-charger/thumbnail.webp, Apple iPhone Charger, 39.98), List(15.54, 219.58, 100, 129.99, 2, https://cdn.dummyjson.com/product-images/mobile-accessories/apple-airpods/thumbnail.webp, Apple Airpods, 259.98))",299.96,2,9,119,null
1074.15,150,2026-07-02T08:27:27.582966,"List(List(12.25, 10.52, 153, 11.99, 1, https://cdn.dummyjson.com/product-images/sports-accessories/volleyball/thumbnail.webp, Volleyball, 11.99), List(13.74, 15.05, 32, 3.49, 5, https://cdn.dummyjson.com/product-images/groceries/milk/thumbnail.webp, Milk, 17.450000000000003), List(3.64, 963.58, 159, 499.99, 2, https://cdn.dummyjson.com/product-images/tablets/ipad-mini-2021-starlight/thumbnail.webp, iPad Mini 2021 Starlight, 999.98))",1029.42,3,8,150,null
696.54,154,2026-07-02T08:27:27.582966,"List(List(12.06, 3.51, 29, 3.99, 1, https://cdn.dummyjson.com/product-images/groceries/juice/thumbnail.webp, Juice, 3.99), List(5.99, 423.0, 93, 89.99, 5, https://cdn.dummyjson.com/product-images/mens-watches/brown-leather-belt-watch/thumbnail.webp, Brown Leather Belt Watch, 449.95), List(11.63, 106.03, 176, 59.99, 2, https://cdn.dummyjson.com/product-images/womens-bags/women-handbag-black/thumbnail.webp, Women Handbag Black, 119.98))",573.92,3,8,154,null
103.65,157,2026-07-02T08:27:27.582966,"List(List(1.53, 4.91, 74, 4.99, 1, https://cdn.dummyjson.com/product-images/kitchen-accessories/spoon/thumbnail.webp, Spoon, 4.99), List(12.62, 1.74, 16, 1.99, 1, https://cdn.dummyjson.com/product-imag

SILVER PIPELINE COMPLETED


In [0]:
%sql 
select * from emain_databricks.silver.users
limit 10


address,age,bank,birthDate,bloodGroup,company,crypto,ein,email,eyeColor,firstName,gender,hair,height,id,image,ingestion_timestamp,ip,lastName,macAddress,maidenName,password,phone,role,ssn,university,userAgent,username,weight,_rescued_data,processing_timestamp,pipeline_name
"List(626 Main Street, Phoenix, List(-77.16213, -92.084824), United States, 29112, Mississippi, MS)",29,"List(05/28, 3693233511855044, Diners Club International, GBP, GB74MH2UZLR9TRPHYNU8F8)",1996-5-30,O-,"List(List(263 Tenth Street, San Francisco, List(71.814525, -161.150263), United States, 37657, Wisconsin, WI), Engineering, Dooley, Kozey and Cronin, Sales Manager)","List(Bitcoin, Ethereum (ERC20), 0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a)",977-175,emily.johnson@x.dummyjson.com,Green,Emily,female,"List(Brown, Curly)",193.24,1,https://dummyjson.com/icon/emilys/128,2026-07-02T08:27:27.575341,42.48.100.32,Johnson,47:fa:41:18:ec:eb,Smith,emilyspass,+81 965-431-3024,admin,900-590-289,University of Wisconsin--Madison,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.93 Safari/537.36",emilys,63.16,null,2026-07-02T09:00:17.426Z,silver_pipeline
"List(385 Fifth Street, Houston, List(22.815468, 115.608581), United States, 38807, Alabama, AL)",36,"List(01/30, 3530633803003665, JCB, USD, DE26362283149158045865)",1989-8-10,B+,"List(List(395 Main Street, Los Angeles, List(79.098326, -119.624845), United States, 73442, New Hampshire, NH), Support, Spinka - Dickinson, Support Specialist)","List(Bitcoin, Ethereum (ERC20), 0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a)",912-602,michael.williams@x.dummyjson.com,Red,Michael,male,"List(Green, Straight)",186.22,2,https://dummyjson.com/icon/michaelw/128,2026-07-02T08:27:27.575341,12.13.116.142,Williams,79:15:78:99:60:aa,,michaelwpass,+49 258-627-6644,admin,108-953-962,Ohio State University,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Edge/97.0.1072.76 Safari/537.36",michaelw,76.32,null,2026-07-02T09:00:17.426Z,silver_pipeline
"List(1642 Ninth Street, Washington, List(45.289366, 46.832664), United States, 32822, Alabama, AL)",43,"List(10/27, 6011212053392887, Discover, EUR, DE12191213468288004835)",1982-11-6,O-,"List(List(1896 Washington Street, Dallas, List(20.086743, -34.577107), United States, 88511, Nevada, NV), Research and Development, Schiller - Zieme, Accountant)","List(Bitcoin, Ethereum (ERC20), 0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a)",963-113,sophia.brown@x.dummyjson.com,Hazel,Sophia,female,"List(White, Wavy)",177.72,3,https://dummyjson.com/icon/sophiab/128,2026-07-02T08:27:27.575341,214.225.51.195,Brown,12:a3:d3:6f:5c:5b,,sophiabpass,+81 210-652-2785,admin,638-461-822,Pepperdine University,"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.45 Safari/537.36",sophiab,52.6,null,2026-07-02T09:00:17.426Z,silver_pipeline
"List(238 Jefferson Street, Seattle, List(16.782513, -139.34723), United States, 68354, Pennsylvania, PA)",46,"List(07/30, 5303440212268149, Mastercard, CAD, DE01300746880579852937)",1979-5-4,AB+,"List(List(1622 Lincoln Street, Fort Worth, List(54.91193, -79.498328), United States, 27768, Pennsylvania, PA), Support, Pagac and Sons, Research Analyst)","List(Bitcoin, Ethereum (ERC20), 0xb9fc2fe63b2a6c003f1c324c3bfa53259162181a)",904-810,james.davis@x.dummyjson.com,Amber,James,male,"List(Blonde, Straight)",193.31,4,https://dummyjson.com/icon/jamesd/128,2026-07-02T08:27:27.575341,101.118.131.66,Davis,10:7d:df:1f:97:58,,jamesdpass,+49 614-958-9364,admin,116-951-314,University of Southern California,"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/97.0.4692.99 Safari/537.36",jamesd,62.1,null,2026-07-02T09:00:17.426Z,silver_pipeline
"List(607 Fourth Street, Jacksonville, List(0.505589, -157.43281), United States, 26593, Colorado, CO)",31,"List(07/30, 5237188057591130, Mastercard, NZD, DE19182355652037133559)",1994-6-13,AB-,"List(

In [0]:
%sql 
select * from emain_databricks.silver.products
limit 10

availabilityStatus,brand,category,description,dimensions,discountPercentage,id,images,ingestion_timestamp,meta,minimumOrderQuantity,price,rating,returnPolicy,reviews,shippingInformation,sku,stock,tags,thumbnail,title,warrantyInformation,weight,_rescued_data,processing_timestamp,pipeline_name
In Stock,Essence,beauty,The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.,"List(22.99, 13.08, 15.14)",10.48,1,List(https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/1.webp),2026-07-02T08:27:27.564321,"List(5784719087687, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",48,11.09,2.56,No return policy,"List(List(Would not recommend!, 2025-04-30T09:41:02.053Z, 3, eleanor.collins@x.dummyjson.com, Eleanor Collins), List(Very satisfied!, 2025-04-30T09:41:02.053Z, 4, lucas.gordon@x.dummyjson.com, Lucas Gordon), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, eleanor.collins@x.dummyjson.com, Eleanor Collins))",Ships in 3-5 business days,BEA-ESS-ESS-001,99,"List(beauty, mascara)",https://cdn.dummyjson.com/product-images/beauty/essence-mascara-lash-princess/thumbnail.webp,Essence Mascara Lash Princess,1 week warranty,4,null,2026-07-02T09:00:05.109Z,silver_pipeline
In Stock,Glamour Beauty,beauty,"The Eyeshadow Palette with Mirror offers a versatile range of eyeshadow shades for creating stunning eye looks. With a built-in mirror, it's convenient for on-the-go makeup application.","List(27.67, 22.47, 9.26)",18.19,2,List(https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/1.webp),2026-07-02T08:27:27.564321,"List(9170275171413, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",20,19.99,2.86,7 days return policy,"List(List(Great product!, 2025-04-30T09:41:02.053Z, 5, savannah.gomez@x.dummyjson.com, Savannah Gomez), List(Awesome product!, 2025-04-30T09:41:02.053Z, 4, christian.perez@x.dummyjson.com, Christian Perez), List(Poor quality!, 2025-04-30T09:41:02.053Z, 1, nicholas.bailey@x.dummyjson.com, Nicholas Bailey))",Ships in 2 weeks,BEA-GLA-EYE-002,34,"List(beauty, eyeshadow)",https://cdn.dummyjson.com/product-images/beauty/eyeshadow-palette-with-mirror/thumbnail.webp,Eyeshadow Palette with Mirror,1 year warranty,9,null,2026-07-02T09:00:05.109Z,silver_pipeline
In Stock,Velvet Touch,beauty,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.","List(20.59, 27.93, 29.27)",10.45,3,List(https://cdn.dummyjson.com/product-images/beauty/powder-canister/1.webp),2026-07-02T08:27:27.564321,"List(8418883906837, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",22,13.64,4.64,No return policy,"List(List(Would buy again!, 2025-04-30T09:41:02.053Z, 4, alexander.jones@x.dummyjson.com, Alexander Jones), List(Highly impressed!, 2025-04-30T09:41:02.053Z, 5, elijah.cruz@x.dummyjson.com, Elijah Cruz), List(Very dissatisfied!, 2025-04-30T09:41:02.053Z, 1, avery.perez@x.dummyjson.com, Avery Perez))",Ships in 1-2 business days,BEA-VEL-POW-003,87,"List(beauty, face powder)",https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp,Powder Canister,3 months warranty,8,null,2026-07-02T09:00:05.109Z,silver_pipeline
In Stock,Chic Cosmetics,beauty,"The Red Lipstick is a classic and bold choice for adding a pop of color to your lips. With a creamy and pigmented formula, it provides a vibrant and long-lasting finish.","List(22.17, 28.38, 18.11)",12.16,4,List(https://cdn.dummyjson.com/product-images/beauty/red-lipstick/1.webp),2026-07-02T08:27:27.564321,"List(9467746727219, 2025-04-30T09:41:02.053Z, https://cdn.dummyjson.com/public/qr-code.png, 2025-04-30T09:41:02.053Z)",40,14.55,4.36,7 days return policy,"List(List

In [0]:
%sql 
select * from emain_databricks.silver.carts
limit 10

discountedTotal,id,ingestion_timestamp,products,total,totalProducts,totalQuantity,userId,_rescued_data,processing_timestamp,pipeline_name
11510.81,1,2026-07-02T08:27:27.582966,"List(List(12.13, 105.41, 162, 29.99, 4, https://cdn.dummyjson.com/product-images/tops/blue-frock/thumbnail.webp, Blue Frock, 119.96), List(12.1, 10547.97, 113, 3999.99, 3, https://cdn.dummyjson.com/product-images/motorcycle/generic-motorcycle/thumbnail.webp, Generic Motorcycle, 11999.97), List(6.69, 839.76, 122, 299.99, 3, https://cdn.dummyjson.com/product-images/smartphones/iphone-6/thumbnail.webp, iPhone 6, 899.97), List(1.71, 17.67, 138, 8.99, 2, https://cdn.dummyjson.com/product-images/sports-accessories/baseball-ball/thumbnail.webp, Baseball Ball, 17.98))",13037.88,4,12,1,null,2026-07-02T09:00:26.774Z,silver_pipeline
125.7,2,2026-07-02T08:27:27.582966,"List(List(6.83, 93.12, 86, 19.99, 5, https://cdn.dummyjson.com/product-images/mens-shirts/man-short-sleeve-shirt/thumbnail.webp, Man Short Sleeve Shirt, 99.94999999999999), List(18.52, 32.58, 104, 19.99, 2, https://cdn.dummyjson.com/product-images/mobile-accessories/apple-iphone-charger/thumbnail.webp, Apple iPhone Charger, 39.98))",139.93,2,8,2,null,2026-07-02T09:00:26.774Z,silver_pipeline
1604.77,3,2026-07-02T08:27:27.582966,"List(List(4.23, 14.36, 24, 14.99, 1, https://cdn.dummyjson.com/product-images/groceries/fish-steak/thumbnail.webp, Fish Steak, 14.99), List(9.37, 996.92, 123, 1099.99, 1, https://cdn.dummyjson.com/product-images/smartphones/iphone-13-pro/thumbnail.webp, iPhone 13 Pro, 1099.99), List(6.95, 279.14, 129, 299.99, 1, https://cdn.dummyjson.com/product-images/smartphones/realme-x/thumbnail.webp, Realme X, 299.99), List(6.83, 93.12, 86, 19.99, 5, https://cdn.dummyjson.com/product-images/mens-shirts/man-short-sleeve-shirt/thumbnail.webp, Man Short Sleeve Shirt, 99.94999999999999), List(14.72, 153.49, 8, 89.99, 2, https://cdn.dummyjson.com/product-images/fragrances/dior-j'adore/thumbnail.webp, Dior J'adore, 179.98), List(11.22, 88.74, 157, 19.99, 5, https://cdn.dummyjson.com/product-images/sunglasses/party-glasses/thumbnail.webp, Party Glasses, 99.94999999999999))",1794.85,6,15,3,null,2026-07-02T09:00:26.774Z,silver_pipeline
636.81,4,2026-07-02T08:27:27.582966,"List(List(0.04, 329.84, 92, 109.99, 3, https://cdn.dummyjson.com/product-images/mens-shoes/sports-sneakers-off-white-red/thumbnail.webp, Sports Sneakers Off White Red, 329.96999999999997), List(14.72, 306.97, 8, 89.99, 4, https://cdn.dummyjson.com/product-images/fragrances/dior-j'adore/thumbnail.webp, Dior J'adore, 359.96))",689.93,2,5,4,null,2026-07-02T09:00:26.774Z,silver_pipeline
1205.8,5,2026-07-02T08:27:27.582966,"List(List(18.2, 1145.17, 161, 349.99, 4, https://cdn.dummyjson.com/product-images/tablets/samsung-galaxy-tab-white/thumbnail.webp, Samsung Galaxy Tab White, 1399.96), List(17.48, 6.57, 39, 1.99, 4, https://cdn.dummyjson.com/product-images/groceries/soft-drinks/thumbnail.webp, Soft Drinks, 7.96), List(9.84, 54.06, 3, 14.99, 4, https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp, Powder Canister, 59.96))",1467.88,3,12,5,null,2026-07-02T09:00:26.774Z,silver_pipeline
46641.06,6,2026-07-02T08:27:27.582966,"List(List(15.94, 46232.96, 192, 10999.99, 5, https://cdn.dummyjson.com/product-images/womens-watches/rolex-datejust-women/thumbnail.webp, Rolex Datejust Women, 54999.95), List(11.51, 309.71, 130, 349.99, 1, https://cdn.dummyjson.com/product-images/smartphones/realme-xt/thumbnail.webp, Realme XT, 349.99), List(4.94, 95.02, 155, 24.99, 4, https://cdn.dummyjson.com/product-images/sunglasses/classic-sun-glasses/thumbnail.webp, Classic Sun Glasses, 99.96), List(14.92, 3.37, 42, 0.99, 4, https://cdn.dummyjson.com/product-images/groceries/water/thumbnail.webp, Water, 3.96))",55453.86,4,19,6,null,2026-07-02T09:00:26.774Z,silver_pipeline
1136.65,7,2026-07-02T08:27:27.582966,"List(List(14.04, 42.97, 56, 49.99, 1, https://cdn.dummyjson.com/product-images/kitchen-accessories/electric-stove/thumbnail.web